In [1]:
#Open 3D Point cloud Viewer




In [1]:
import open3d as o3d
import os
import numpy as np
import glob
import sys

# ======================= CONFIGURATION =======================
# !!! ABSOLUTE PATH CONFIRMED BY USER !!!
# This path is where 'seq_01', 'seq_02', and 'seq_03' folders are located.
BASE_PROJECT_PATH = "/home/ahzsyed/Documents/GitHub/Perception12/Final_Project/results/disparity_depth"

# Define the sequence folders to check
SEQUENCE_IDS = ["seq_01", "seq_02", "seq_03"]

# Define the subfolder where the PLY files reside (e.g., 'pointcloud')
POINT_CLOUD_SUBDIR = "pointcloud"

# Define the colors for each sequence for easy visual differentiation (RGB values 0.0 to 1.0)
SEQUENCE_COLORS = {
    "seq_01": [1.0, 0.0, 0.0],  # Red
    "seq_02": [0.0, 1.0, 0.0],  # Green
    "seq_03": [0.0, 0.0, 1.0],  # Blue
}

# Base output directory for screenshots
SCREENSHOT_DIR = "output_screenshots"
# =============================================================

def find_ply_file(sequence_path):
    """
    Looks for the first .ply file inside the sequence's pointcloud subfolder.
    """
    # Create the full search path using glob wildcard
    search_path = os.path.join(sequence_path, POINT_CLOUD_SUBDIR, "*.ply")
    print(f"DEBUG: Full search path: {search_path}")
    
    ply_files = glob.glob(search_path)
    if ply_files:
        # Returns the path to the first PLY file found
        return ply_files[0]
    return None

def load_and_colorize_pcd(file_path, color):
    """
    Loads a point cloud and assigns a uniform color if it doesn't have one,
    or uses the existing color data if present.
    """
    if not os.path.exists(file_path):
        print(f"Warning: File not found at '{file_path}'. Skipping.")
        return None

    try:
        pcd = o3d.io.read_point_cloud(file_path)
        
        if not pcd.has_points():
            print(f"Warning: Point cloud in '{file_path}' has no points. Skipping.")
            return None
        
        # Assign the specified color if the point cloud has no existing color data
        if not pcd.has_colors():
            # Create a color array (N, 3) where N is the number of points
            colors_array = np.asarray([color] * len(pcd.points), dtype=np.float64)
            pcd.colors = o3d.utility.Vector3dVector(colors_array)
        else:
            print(f"Note: {os.path.basename(file_path)} already has color data, using existing colors.")

        return pcd

    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

def visualize_and_save(seq_id, pcd):
    """
    Opens a separate visualization window for a single point cloud and saves a screenshot upon closing.
    """
    
    screenshot_filename = f"{seq_id}_pointcloud.png"
    save_path = os.path.join(SCREENSHOT_DIR, screenshot_filename)

    try:
        # Create a visualizer instance
        vis = o3d.visualization.Visualizer()
        vis.create_window(window_name=f"Point Cloud: {seq_id} - Close to Save Screenshot", width=1024, height=768)
        
        # Add the geometry
        vis.add_geometry(pcd)

        # Optional: Set a good initial view
        view_control = vis.get_view_control()
        view_control.set_up([0, -1, 0]) # Y-up
        view_control.set_zoom(0.8) # Adjust zoom if needed
        
        # Run the visualizer. Execution blocks here until the user closes the window.
        vis.run()

        # Save the screenshot after closing the window
        if not os.path.isdir(SCREENSHOT_DIR):
            os.makedirs(SCREENSHOT_DIR, exist_ok=True)
            
        vis.capture_screen_image(save_path)
        print(f"  [INFO] Screenshot saved successfully for {seq_id} to: {save_path}")

        vis.destroy_window()

    except Exception as e:
        print(f"An error occurred during visualization for {seq_id}: {e}")
        print("Ensure you have a valid graphical environment (e.g., X server) if running remotely.")
        sys.exit(1)


def view_individual_sequence_point_clouds():
    """
    Loads point clouds sequentially from the defined sequence folders and displays them individually.
    """
    if not os.path.isdir(BASE_PROJECT_PATH):
        print("\n--- FATAL ERROR ---")
        print(f"The configured base path does not exist or is not a directory: {BASE_PROJECT_PATH}")
        print("Please verify the absolute path on your system.")
        print("-------------------\n")
        return

    loaded_sequences = []
    
    print("--- Starting Individual Open3D Point Cloud Viewer ---")
    
    # 1. Iterate through each sequence ID
    for seq_id in SEQUENCE_IDS:
        sequence_folder_path = os.path.join(BASE_PROJECT_PATH, seq_id)
        
        print(f"\nSearching for PLY in sequence folder: {sequence_folder_path}")
        
        # 2. Find the actual PLY file inside the subfolder
        ply_file_path = find_ply_file(sequence_folder_path)
        
        if ply_file_path:
            print(f"Found file: {os.path.basename(ply_file_path)}")
            color = SEQUENCE_COLORS.get(seq_id, [0.5, 0.5, 0.5])
            
            # 3. Load and colorize the point cloud
            pcd = load_and_colorize_pcd(ply_file_path, color)
            
            if pcd:
                loaded_sequences.append(seq_id)
                print(f"  [SUCCESS] Loaded {seq_id} (Points: {len(pcd.points)}). Ready for visualization.")
                
                # 4. Visualize and save the individual point cloud
                visualize_and_save(seq_id, pcd)
        else:
            print(f"Warning: No .ply file found for {seq_id}. Skipping visualization for this sequence.")


    if not loaded_sequences:
        print("\nNo valid point clouds were loaded or visualized.")
    else:
        print("\n--- Summary ---")
        print(f"Successfully loaded, visualized, and saved screenshots for the following sequences: {', '.join(loaded_sequences)}")
        print(f"Screenshots saved to the '{SCREENSHOT_DIR}/' directory.")

if __name__ == "__main__":
    view_individual_sequence_point_clouds()

--- Starting Individual Open3D Point Cloud Viewer ---

Searching for PLY in sequence folder: /home/ahzsyed/Documents/GitHub/Perception12/Final_Project/results/disparity_depth/seq_01
DEBUG: Full search path: /home/ahzsyed/Documents/GitHub/Perception12/Final_Project/results/disparity_depth/seq_01/pointcloud/*.ply
Found file: 000057.ply
Note: 000057.ply already has color data, using existing colors.
  [SUCCESS] Loaded seq_01 (Points: 333182). Ready for visualization.
  [INFO] Screenshot saved successfully for seq_01 to: output_screenshots/seq_01_pointcloud.png

Searching for PLY in sequence folder: /home/ahzsyed/Documents/GitHub/Perception12/Final_Project/results/disparity_depth/seq_02
DEBUG: Full search path: /home/ahzsyed/Documents/GitHub/Perception12/Final_Project/results/disparity_depth/seq_02/pointcloud/*.ply
Found file: 0000000200.ply
Note: 0000000200.ply already has color data, using existing colors.
  [SUCCESS] Loaded seq_02 (Points: 350965). Ready for visualization.
  [INFO] Scre